In [1]:
!pip install cellphonedb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 36.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 36.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00:00:01


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 23.7 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 MB 30.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 6.5 MB/s eta 0:00:00
  Created wheel for numpy-groupies: filename=numpy_groupies-0.9.22-py3-none-any.whl size=25846 sha256=d6f8c94510f56caf2071bdd6023e6a6c2f686d1a1c4c3de6c8c4c4c03f0b35c6
  Stored in directory: /pfs/lustrep1/users/wangjun1/.cache/pip/wheels/e7/be/61/60c3bfac3b63cbe691ea4a6a8d9acdeaf2670ed3cd439ecb2a
  Created wheel for fbpca: filename=fbpca-1.0-py3-none-any.whl size=11373 sha256=18b9dc9deae476a8990106722b669f94f665f1895df41ef0dd92a53cf0b2d157
  Stored in directory: /pfs/lustrep1/users/wangjun1/.cache/pip/wheels/b4/3b/77/a06a07a415b222f47a7e522333f85ce64c0defd07a57762267
Successfully built numpy-groupies fbpca
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully unins

# Download the dataset from source

In [2]:
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils

display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

In [4]:
import os
# -- Version of the databse
cpdb_version = 'v5.0.0'
# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join('/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database', cpdb_version)

In [5]:
from cellphonedb.utils import db_utils

db_utils.download_database(cpdb_target_dir, cpdb_version)

Downloaded cellphonedb.zip into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0
Downloaded complex_input.csv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0
Downloaded gene_input.csv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0
Downloaded interaction_input.csv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0
Downloaded protein_input.csv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0
Downloaded uniprot_synonyms.tsv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0/sources
Downloaded transcription_factor_input.csv into /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_database/v5.0.0/sources


# Loading necessary files

In [2]:
import scanpy as sc
#index the embeddings by cell_ids
data_dir = "/scratch/project_465001027/spatialformer/david_data/relabel_output-XETG00048__0003392__THD0008__20230313__191400/outs/cell_feature_matrix"
adata = sc.read_10x_mtx(data_dir,  # The directory containing the files
                            var_names='gene_symbols',  # Use 'gene_ids' if you prefer the IDs
                            cache=True) 

adata

AnnData object with n_obs × n_vars = 73785 × 343
    var: 'gene_ids', 'feature_types'

adding the annotation data

In [5]:
import pickle
from datasets import load_from_disk
import sys
sys.path.append("/scratch/project_465001027/Spatialformer/utils")
from utils import *

#loading the 
combined_dataset = load_from_disk("/scratch/project_465001027/Spatialformer/cache/xenium_pandavid_dataset4")  
index_path = "/scratch/project_465001027/Spatialformer/data/sample_cell_index.pkl"
sample_cell_index = get_index(combined_dataset, save_file = index_path)
sample_name = "THD0008"

combined_dataset_all = concatenate_datasets([combined_dataset["train"], combined_dataset["test"], combined_dataset["validation"]])
sample_dataset = combined_dataset_all.select(list(sample_cell_index[sample_name].values()))

Loading dataset from disk:   0%|          | 0/364 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/114 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/91 [00:00<?, ?it/s]

In [9]:
kept_id = [cell_id for cell_id, index in sample_cell_index[sample_name].items()]
# kept_index = [index for cell_id, index in sample_cell_index[sample_name].items()]
# anns = combined_dataset_all.select(kept_index)["Annotations"]

In [10]:
adata = adata[np.isin(adata.obs.index, kept_id)]

In [12]:
index = [sample_cell_index[sample_name][cell_id] for cell_id in adata.obs.index]
anns = combined_dataset_all.select(index)["Annotations"]

In [13]:
adata.obs["Annottions"] = anns

/tmp/ipykernel_68430/2092349314.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["Annottions"] = anns


In [14]:
adata

AnnData object with n_obs × n_vars = 57012 × 343
    obs: 'Annottions'
    var: 'gene_ids', 'feature_types'

saving the count matrix and metadata

In [ ]:
adata.write("/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/normalised_log_counts.h5ad")


In [15]:
adata.obs.to_csv('/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/metadata.tsv', sep='\t', index=True)

,Annottions
aaaaaaab-1,gCap
aaaaaaac-1,AT2
aaaaaaae-1,NK cells
aaaaaaag-1,gCap
aaaaaaah-1,Fibroblasts
...,...
aaabcada-1,Proliferating Epithelial
aaabcadc-1,Proliferating Endothelial
aaabcadd-1,gCap
aaabcadf-1,Macrophages


In [ ]:
cpdb_file_path = 'db/v5/cellphonedb.zip'
meta_file_path = 'data/metadata.tsv'
counts_file_path = 'data/normalised_log_counts.h5ad'
out_path = '/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/cpdb_results'